# סורק Momentum — Google Colab

**לא מעלים כלום ל-Colab.** הקבצים כבר ב-Google Drive.

הרץ תאים **לפי סדר** (1 → 2 → 3 → 4 → 5).

תיקייה ב-Drive: `סורק מומונטום חדש` (או `momentum_system`) — תא 3 מוצא אוטומטית.

רמת סריקה בתא 4: `simple` | `medium` | `full`


In [ ]:
!pip install -q pandas "numpy>=1.26" pyarrow python-dateutil pytz
!pip install -q tiingo requests urllib3 ratelimit tenacity
!pip install -q pyyaml python-dotenv
!pip install -q streamlit altair
!pip install -q rich loguru
!pip install -q "pandas-ta"
!pip install -q pyngrok
print("התקנה הושלמה.")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")

CANDIDATES = [
    DRIVE / "סורק מומונטום חדש" / "momentum_system",
    DRIVE / "סורק מומונטום חדש",
    DRIVE / "momentum_system",
]


def is_project_root(p: Path) -> bool:
    return (p / "scripts" / "run_pro_scanner.py").exists()


def find_project_root() -> Path | None:
    checked = set()
    extra = []
    if DRIVE.exists():
        for folder in DRIVE.iterdir():
            if not folder.is_dir():
                continue
            name = folder.name
            if "מומנטום" in name or "momentum" in name.lower() or "סורק" in name:
                extra.append(folder)
                extra.append(folder / "momentum_system")
    for cand in CANDIDATES + extra:
        try:
            c = cand.resolve()
        except OSError:
            continue
        if c in checked:
            continue
        checked.add(c)
        if is_project_root(c):
            return c
        nested = c / "momentum_system"
        if nested not in checked and is_project_root(nested):
            return nested
    return None


root = find_project_root()
if root is None:
    print("לא מצאתי את הפרויקט. תיקיות ב-Drive:")
    for p in sorted(DRIVE.iterdir(), key=lambda x: x.name):
        if p.is_dir():
            print(" -", p.name)
    raise FileNotFoundError(
        "שים את התיקייה עם scripts/run_pro_scanner.py תחת Drive"
    )

os.chdir(root)
print("עובד מתוך:", os.getcwd())


In [ ]:
import os

os.environ["DATA_PROVIDER"] = "polygon"
os.environ["POLYGON_API_KEY"] = "הדבק_כאן_מפתח_Polygon"
os.environ["DASHBOARD_PASSWORD"] = "סיסמה_לכניסה"
os.environ["SCANNER_UNIVERSE_CSV"] = "data/universe/polygon_liquid_us.csv"
os.environ["SCANNER_SECTOR_MAP"] = "data/universe/sector_map.csv"
# רמת סריקה: simple (מהיר) | medium (מאוזן) | full (מקיף)
os.environ["SCAN_PROFILE"] = "simple"
os.environ["ENABLE_DASHBOARD_SCAN_BUTTON"] = "true"

if "הדבק" in os.environ["POLYGON_API_KEY"]:
    raise ValueError("הזן POLYGON_API_KEY אמיתי")
print("סודות הוגדרו. רמת סריקה:", os.environ["SCAN_PROFILE"])


In [ ]:
import os, subprocess, sys
profile = os.environ.get('SCAN_PROFILE', 'simple')
print('מריץ סריקה:', profile)
subprocess.run([
    sys.executable, 'scripts/run_pro_scanner.py',
    '--universe-csv', 'data/universe/polygon_liquid_us.csv',
    '--sector-map', 'data/universe/sector_map.csv',
    '--profile', profile,
], check=True)


### סריקה ברמה אחרת
שנה `SCAN_PROFILE` בתא 4 ל-`medium` או `full` והרץ שוב את תא 5.


### אחרי סריקה — העלאה לאתר (קישור מהמייל)
הרץ את התא הבא עם אסימון Write מ-Hugging Face (פעם אחת).  
אז מכל מחשב בעולם תפתח את הקישור במייל ותראה את **כל** 2,114 המניות.

In [ ]:
# אופציונלי: העלאת הדוח האחרון ל-Hugging Face Space
import os
from pathlib import Path

HF_TOKEN = "הדבק_אסימון_Write_מ_HuggingFace"  # https://huggingface.co/settings/tokens
HF_USER = "azr6768689"
HF_SPACE = "momentum-scanner"

if "הדבק" in HF_TOKEN:
    print("דלג — הזן HF_TOKEN כדי להעלות לאתר מהמייל")
else:
    reports = sorted(Path("data/reports").glob("*_report.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not reports:
        raise FileNotFoundError("אין דוח — הרץ קודם את תא הסריקה")
    latest = reports[0]
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    os.environ["HF_USERNAME"] = HF_USER
    os.environ["HF_SPACE_NAME"] = HF_SPACE
    !pip install -q huggingface_hub
    !python scripts/upload_report_to_hf.py "{latest}"
    print(f"https://huggingface.co/spaces/{HF_USER}/{HF_SPACE}")

In [ ]:
# דוגמה: סריקה מקיפה ידנית
# import subprocess, sys
# subprocess.run([sys.executable, 'scripts/run_pro_scanner.py',
#   '--universe-csv', 'data/universe/polygon_liquid_us.csv', '--profile', 'full'], check=True)


In [ ]:
import subprocess, sys, time
from pyngrok import ngrok

port = 8501
subprocess.Popen([
    sys.executable, "-m", "streamlit", "run", "dashboard/app.py",
    "--server.port", str(port), "--server.headless", "true",
])
time.sleep(10)
print("פתח:", ngrok.connect(port))